# Crypto Narrative & NLP: offline research pipeline

Run all cells from repository root or notebooks/. Inputs: data/processed/market_daily.parquet and text_daily.parquet. No network calls. If real text is absent, the notebook explicitly uses seeded synthetic fixtures and labels every artifact accordingly. Never mix real prices with synthetic text.

Required environment: project requirements plus matplotlib, nbformat, nbclient and ipykernel. Existing project .venv packages can be reused by the host Jupyter kernel.


In [1]:
from pathlib import Path
import sys, json, hashlib, subprocess
import pandas as pd
import yaml
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
assert (root / 'config/config.yaml').is_file(), 'Run from repository root or notebooks/'
sys.path.insert(0, str(root))
# Reuse installed project research packages with the host notebook kernel.
project_packages = root / '.venv' / ('Lib/site-packages' if sys.platform == 'win32' else f'lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages')
if project_packages.exists():
    sys.path.insert(1, str(project_packages))
from src.models.run import run_analysis
config = yaml.safe_load((root/'config/config.yaml').read_text(encoding='utf-8'))
real = (root/'data/processed/text_daily.parquet').is_file()
if real:
    market = pd.read_parquet(root/'data/processed/market_daily.parquet')
    text = pd.read_parquet(root/'data/processed/text_daily.parquet')
else:
    import importlib.util
    spec = importlib.util.spec_from_file_location('fixture_data', root/'tests/fixtures/make_fixtures.py')
    fixture_data = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(fixture_data)
    market, text = fixture_data.make_fixtures(config)
mode = 'real' if real else 'synthetic'
print(mode.upper(), 'market rows', len(market), 'text rows', len(text))
ucry_path = root/'data/processed/ucry_daily.parquet'
ucry = pd.read_parquet(ucry_path) if real and ucry_path.exists() else None

from src.evaluation.hourly_event import hourly_prices
interim = root/'data/interim'
event_inputs = (pd.read_parquet(interim/'documents.parquet'), pd.read_parquet(interim/'doc_scores.parquet'), hourly_prices()) if real and (interim/'documents.parquet').exists() and (interim/'doc_scores.parquet').exists() else None


REAL market rows 6845 text rows 2126


## Recompute all tables and paper figures

Core spec uses only same-day text; market return lags remain available on sparse news dates. Real runs also read cached hourly Binance bars and canonical articles for event tests. Tercile cutoffs use training articles only. Every refit purges horizon plus embargo; no network calls.


In [2]:
output = root/'outputs/tables'/mode
figures = root/'outputs/figures'/mode
tables = run_analysis(market, text, config, output, figures, real=real, ucry=ucry, event_inputs=event_inputs)
print('Written:', output, figures)
inputs = [root/'data/processed/market_daily.parquet', root/'data/processed/text_daily.parquet']
if ucry is not None:
    inputs.append(ucry_path)
if event_inputs is not None:
    inputs += [interim/'documents.parquet', interim/'doc_scores.parquet']
    inputs += sorted((root/'data/raw').glob('*_1h_*.parquet'))
code_paths = [root/'src/models/run.py', root/'src/models/ml.py', root/'src/evaluation/report.py', root/'src/evaluation/hourly_event.py']
metadata = {'result_source': mode, 'config': config, 'refit_days': 30,
            'pipeline_commit': subprocess.check_output(['git','rev-parse','HEAD'], cwd=root, text=True).strip(),
            'sha256': {str(path): hashlib.sha256(path.read_bytes()).hexdigest() for path in inputs},
            'analysis_code_sha256': hashlib.sha256(b''.join(path.read_bytes() for path in code_paths)).hexdigest()}
(output/'run_metadata.json').write_text(json.dumps(metadata, indent=2)+'\n', encoding='utf-8')


Running econometrics


Running robustness


C:\Users\Dell\crypto-narrative-nlp\src\models\econometrics.py:30: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(data[target], sm.add_constant(data[features], has_constant='add')).fit(
C:\Users\Dell\crypto-narrative-nlp\src\models\econometrics.py:30: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(data[target], sm.add_constant(data[features], has_constant='add')).fit(
C:\Users\Dell\crypto-narrative-nlp\src\models\econometrics.py:30: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(data[target], sm.add_constant(data[features], has_constant='add')).fit(


C:\Users\Dell\crypto-narrative-nlp\src\models\econometrics.py:30: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(data[target], sm.add_constant(data[features], has_constant='add')).fit(
C:\Users\Dell\crypto-narrative-nlp\src\models\econometrics.py:30: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(data[target], sm.add_constant(data[features], has_constant='add')).fit(
C:\Users\Dell\crypto-narrative-nlp\src\models\econometrics.py:30: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(data[target], sm.add_constant(data[features], has_constant='add')).fit(


C:\Users\Dell\crypto-narrative-nlp\src\models\econometrics.py:30: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(data[target], sm.add_constant(data[features], has_constant='add')).fit(
C:\Users\Dell\crypto-narrative-nlp\src\models\econometrics.py:30: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(data[target], sm.add_constant(data[features], has_constant='add')).fit(
C:\Users\Dell\crypto-narrative-nlp\src\models\econometrics.py:30: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit = sm.OLS(data[target], sm.add_constant(data[features], has_constant='add')).fit(


...

ML: XRP, h=7, target=ret


ML: XRP, h=7, target=rv


Running hourly events


Written: C:\Users\Dell\crypto-narrative-nlp\outputs\tables\real C:\Users\Dell\crypto-narrative-nlp\outputs\figures\real


3626

In [3]:
display(tables['coverage'])
for name in ['regressions','wald','quantiles','granger','oos','gw','mcs','ml_status']:
    print(name, len(tables[name]), 'rows')
    display(tables[name].head(10))
audit = tables['refit_audit']
if not audit.empty:
    assert (audit.train_target_max + pd.to_timedelta(audit.horizon+audit.embargo, unit='D') < audit.origin).all()
print((output/'README.md').read_text(encoding='utf-8'))

if real:
    for name in ['event_coverage', 'event_tests', 'event_asymmetry', 'event_timing']:
        print(name, len(tables.get(name, [])), 'rows')
        display(tables.get(name, pd.DataFrame()).head(10))
    print((output/'RESULTS.md').read_text(encoding='utf-8'))


,asset,calendar_days,news_days,pct_days_with_news,usable_model_days,finbert_days,pct_days_with_finbert,model_development_days,model_test_days,model_train_days,model_valid_days,news_start,news_end,model_start,model_end,configured_start,configured_end
0,BNB,1369,180,13.148283,138,180,13.148283,111,27,101,10,2023-01-01 00:00:00+00:00,2026-09-23 00:00:00+00:00,2023-01-01 00:00:00+00:00,2026-09-23 00:00:00+00:00,2023-01-01,2026-09-30
1,BTC,1369,180,13.148283,176,180,13.148283,142,34,130,12,2023-01-01 00:00:00+00:00,2026-09-23 00:00:00+00:00,2023-01-01 00:00:00+00:00,2026-09-23 00:00:00+00:00,2023-01-01,2026-09-30
2,ETH,1369,180,13.148283,169,180,13.148283,139,30,127,12,2023-01-01 00:00:00+00:00,2026-09-23 00:00:00+00:00,2023-01-01 00:00:00+00:00,2026-09-23 00:00:00+00:00,2023-01-01,2026-09-30
3,SOL,1369,180,13.148283,144,180,13.148283,120,24,111,9,2023-01-01 00:00:00+00:00,2026-09-23 00:00:00+00:00,2023-01-01 00:00:00+00:00,2026-09-23 00:00:00+00:00,2023-01-01,2026-09-30
4,XRP,1369,180,13.148283,151,180,13.148283,122,29,110,12,2023-01-01 00:00:00+00:00,2026-09-23 00:00:00+00:00,2023-01-01 00:00:00+00:00,2026-09-23 00:00:00+00:00,2023-01-01,2026-09-30


regressions 4815 rows


,asset,horizon,target,sentiment,version,term,coef,se,pvalue,identified,nobs,resid_df,r2,sample_start,sample_end,target_max,p_holm,p_bh
0,BNB,1,ret,none,market,const,0.054232,0.100053,0.587797,True,111.0,100.0,0.089717,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,2025-12-25 00:00:00+00:00,1.0,0.845100
1,BNB,1,ret,none,market,ret,0.008226,0.127250,0.948457,True,111.0,100.0,0.089717,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,2025-12-25 00:00:00+00:00,1.0,0.986200
2,BNB,1,ret,none,market,rv,5.488833,4.866440,0.259364,True,111.0,100.0,0.089717,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,2025-12-25 00:00:00+00:00,1.0,0.637366
3,BNB,1,ret,none,market,logvol,-0.003042,0.005495,0.579815,True,111.0,100.0,0.089717,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,2025-12-25 00:00:00+00:00,1.0,0.841752
4,BNB,1,ret,none,market,ret_lag1,0.006481,0.104097,0.950356,True,111.0,100.0,0.089717,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,2025-12-25 00:00:00+00:00,1.0,0.986200
5,BNB,1,ret,none,market,ret_lag2,0.161268,0.080880,0.046161,True,111.0,100.0,0.089717,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,2025-12-25 00:00:00+00:00,1.0,0.293612
6,BNB,1,ret,none,market,ret_lag3,-0.041789,0.122522,0.733048,True,111.0,100.0,0.089717,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,2025-12-25 00:00:00+00:00,1.0,0.912926
7,BNB,1,ret,none,market,ret_lag4,0.030956,0.083854,0.712006,True,111.0,100.0,0.089717,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,2025-12-25 00:00:00+00:00,1.0,0.907326
8,BNB,1,ret,none,market,ret_lag5,0.031327,0.088971,0.724763,True,111.0,100.0,0.089717,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,2025-12-25 00:00:00+00:00,1.0,0.910199
9,BNB,1,ret,none,market,ret_lag6,0.014524,0.080198,0.856286,True,111.0,100.0,0.089717,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,2025-12-25 00:00:00+00:00,1.0,0.958455


wald 15 rows


,asset,horizon,beta_neg,beta_pos,identified,nobs,resid_df,sample_start,sample_end,statistic,pvalue,p_holm,p_bh
0,BNB,1,0.000256,-0.000091,True,111.0,89.0,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,1.785626,0.181460,1.000000,0.497336
1,BTC,1,0.001740,0.001038,True,142.0,120.0,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,1.161277,0.281201,1.000000,0.602573
2,ETH,1,0.000131,0.000026,True,139.0,117.0,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,0.061800,0.803673,1.000000,0.861079
3,SOL,1,0.001108,-0.000742,True,120.0,98.0,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,6.263604,0.012324,0.172541,0.092432
4,XRP,1,0.022523,0.003009,True,122.0,100.0,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,1.964355,0.161048,1.000000,0.497336
5,BNB,3,-0.000100,-0.000277,True,110.0,88.0,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,0.195353,0.658498,1.000000,0.861079
6,BTC,3,-0.001026,-0.001521,True,142.0,120.0,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,0.520545,0.470609,1.000000,0.861079
7,ETH,3,0.001552,-0.000577,True,139.0,117.0,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,11.202263,0.000817,0.012255,0.012255
8,SOL,3,0.002850,0.001266,True,120.0,98.0,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,2.663105,0.102700,1.000000,0.497336
9,XRP,3,-0.003900,-0.004413,True,122.0,100.0,2023-01-01 00:00:00+00:00,2025-12-24 00:00:00+00:00,0.081486,0.775294,1.000000,0.861079


...

event_asymmetry 24 rows


,tone,window,metric,n_days,negative_mean,positive_mean,negative_minus_positive,p_t,p_wilcoxon,p_t_holm,p_wilcoxon_holm
0,finbert_tercile,"[0,1]",car,215,-0.000102,0.000069,-0.000171,0.297364,0.661484,1.000000,1.000000
1,finbert_tercile,"[0,1]",absret,215,-0.000195,0.000214,-0.000409,0.001460,0.019133,0.030653,0.306135
2,finbert_tercile,"[0,1]",volume,215,-0.010682,0.027942,-0.038623,0.004148,0.004975,0.078819,0.104472
3,finbert_tercile,"[0,4]",car,215,-0.000474,-0.000293,-0.000181,0.433111,0.912556,1.000000,1.000000
4,finbert_tercile,"[0,4]",absret,215,-0.001000,-0.000311,-0.000690,0.003038,0.069596,0.060756,0.904752
5,finbert_tercile,"[0,4]",volume,215,-0.050864,-0.005954,-0.044910,0.000153,0.000069,0.003680,0.001660
6,finbert_tercile,"[0,24]",car,214,-0.001244,-0.001264,0.000020,0.966968,0.518305,1.000000,1.000000
7,finbert_tercile,"[0,24]",absret,214,-0.002886,-0.001225,-0.001661,0.038170,0.199622,0.458039,1.000000
8,finbert_tercile,"[0,24]",volume,214,-0.016975,0.002943,-0.019917,0.004865,0.007664,0.087563,0.145620
9,finbert_tercile,"[-4,0]",car,215,0.000208,-0.000105,0.000313,0.182856,0.014960,1.000000,0.256199


event_timing 3 rows


,window,n_pairs,n_days,n_unique_articles,mean_car_published,mean_car_seen,mean_published_minus_seen,p_paired_t,p_paired_wilcoxon,p_public,p_seen,significance_changed,p_paired_t_holm,p_paired_wilcoxon_holm,p_public_holm,p_seen_holm,significance_changed_holm
0,0_1,39,32,37,0.000400,-0.001251,0.001651,0.223323,0.379427,0.761871,0.093490,False,0.446646,0.758853,0.764062,0.186979,False
1,0_4,39,32,37,-0.001608,-0.001915,0.000307,0.889559,0.875446,0.382031,0.223559,False,0.889559,0.875446,0.764062,0.223559,False
2,0_24,39,32,37,-0.004633,-0.010213,0.005579,0.066979,0.102352,0.228964,0.013545,True,0.200937,0.307055,0.686892,0.040634,True


# Kết quả thật, bộ tin đang thu

Bằng chứng dưới đây chỉ áp dụng cho dữ liệu đã thu. Tin cùng ngày được gom theo ngày khi kiểm định sự kiện.

Coverage: 180/1369 ngày có tin (13.15%); từ 2023-01-01 00:00:00+00:00 đến 2026-09-23 00:00:00+00:00. Core spec dùng 176 ngày BTC, train/valid/test=130/12/34.

Sự kiện giờ: 37179 tin canonical tiếng Anh, 39003 cặp tin–asset trên 772 ngày; 18738 tin có FinBERT. Mẫu 2023-01-01 02:00:00+00:00 đến 2026-09-30 21:00:00+00:00.

Bản documents hiện phủ 339 ngày GKG và 503 ngày SEC; text_daily có thể chưa được build lại trên cùng ảnh chụp dữ liệu.

Phản ứng chung [0,4]h car: 0.03715 %, 772 ngày, p t Holm=1.
Phản ứng chung [0,4]h absret: 0.03134 %, 772 ngày, p t Holm=1.
Phản ứng chung [0,4]h volume: 0.03151 log điểm, 772 ngày, p t Holm=1.
Abnormal volume là chênh log(1+USD volume) so với 144 giờ ước lượng trước tin; abnormal |return| là chênh tổng độ lớn lợi suất log.

## RQ1: FinBERT và từ điển

Đã có: hệ số BTC h=1 sent_lm=-0.02807, Holm p=1 (n=143); sent_

## Interpretation

Read `outputs/tables/real/RESULTS.md` for RQ1?RQ5 and `README.md` for coverage and each table. Hourly events cluster by day. The 24-hour placebo is omitted where a ?24h shift would overlap T0. Synthetic findings validate code only; strategy plots are economic-value tests.
